# Phase 4: MLP search on Colab

Runtime → Change runtime type → **T4 GPU**. Upload the folder `phase4` to the root of your Google Drive with:
`X_pool.npy, y_pool.npy, fold_pool.npy, ident_pool.npy, annotated_pool.npy, X_test.npy, y_test.npy, ident_test.npy, meta.json` (from `data/colab_pack/`) and `mlp_search.py`.

Every finished fold is saved to `MyDrive/phase4/results`. If the session is cut off, reconnect and run the cells again: finished folds are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/phase4'
OUT = DRIVE + '/results'
!mkdir -p /content/pack "$OUT"
!cp -n "$DRIVE"/*.npy "$DRIVE"/meta.json /content/pack/
!cp "$DRIVE"/mlp_search.py /content/
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Quick check (about 1 minute)
A tiny run on 30,000 rows to confirm everything works. Writes to a scratch folder, not the results.

In [ ]:
!python /content/mlp_search.py stage1 --pack /content/pack --out /content/smoke --limit 30000 --max-epochs 2 | tail -3

## Stage 1: loss, architecture, text features (15 settings × 5 folds)

In [ ]:
!python /content/mlp_search.py stage1 --pack /content/pack --out "$OUT"

## Stage 2: activation, dropout, learning rate, batch size, L2, identity weights (10 settings around the stage-1 winner)

In [ ]:
!python /content/mlp_search.py stage2 --pack /content/pack --out "$OUT"

## One 6-output network vs six single-label networks (best setting)

In [ ]:
!python /content/mlp_search.py separate --pack /content/pack --out "$OUT"

## Final model: 5 seeds on all pool rows, cuts from out-of-fold scores, test used once

In [ ]:
!python /content/mlp_search.py final --pack /content/pack --out "$OUT"

## Progress

In [ ]:
import json, collections
rows = [json.loads(l) for l in open(OUT + '/results.jsonl')]
c = collections.Counter(r['stage'] for r in rows)
print(c, 'folds done;', sum(r['seconds'] for r in rows) / 3600, 'GPU hours')